# ADAM Simulations - NonConvex

## Parameters

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

In [3]:
from solvers import NonlocalSolverMomentumAdam, AdamMomentum
from sklearn.model_selection import ParameterGrid
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import numpy as np

# Hyperparameter grid for Adam:
# - Two learning rates (lr)
# - Two values for beta1 (first-moment decay)
# - Two values for beta2 (second-moment decay)
param_grid = {'lr': [0.1, 0.01], 'beta1': [0.0, 0.9], 'beta2': [0.99, 0.999]}
n_learning_rates = len(param_grid['lr'])

# Expand the grid into a list of dicts with all combinations
# e.g., [{'lr': 0.1, 'beta1': 0.0, 'beta2': 0.99}, ...]
param_list = list(ParameterGrid(param_grid))

# Problem setup:
# Gradient dL(y) = y (y^2 − 1), i.e., derivative of (1/4)(y^2 − 1)^2
dL = lambda y: y * (y**2 - 1)
f = lambda x, y: 0.0

# Create output folder for figures if it does not exist
figures_dir = "figures"
os.makedirs(figures_dir, exist_ok=True)


[Solver] Using JAX in: NVIDIA GeForce RTX 5070 Ti (gpu)
[Solver] Using JAX in: NVIDIA GeForce RTX 5070 Ti (gpu)


## Adam - Discrete

In [4]:
# Adam - Discrete (JAX) | Save PNG per initial condition
inits = [0.1, 0, -0.1]

for theta_initial in inits:
    # Create figures for this initial condition
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_m = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    for i, lr in enumerate(param_grid['lr']):
        # Filter all parameter configs matching this lr
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs per LR 
        epochs = 100 if lr == 0.1 else 200

        # Run one simulation per configuration at this lr
        for params in filtered_params:
            print(f'\nAdam Configuration: {params}, theta_initial={theta_initial}')
            solver = AdamMomentum(
                dL=dL, lr=lr, beta1=params['beta1'], beta2=params['beta2'], epochs=epochs
            )
            solver.solve(theta_initial=theta_initial)

            label = f"theta_initial={theta_initial}, beta1={params['beta1']}, beta2={params['beta2']}"

            # Theta_k trajectory
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='lines',
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            # First moment m_k
            fig_m.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.m_result,
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            # Second moment v_k
            fig_v.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.v_result,
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

    # Layouts for this initial condition
    fig_theta.update_layout(
        title_text=f'Theta convergence (Adam) — theta_initial={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_theta.update_xaxes(title_text="k")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta_k")

    fig_m.update_layout(
        title_text=f'First moment m (Adam) — theta_initial={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_m.update_xaxes(title_text="k")
    fig_m.update_yaxes(tickformat=".2f", title_text="m_k")

    fig_v.update_layout(
        title_text=f'Second moment v (Adam) — theta_initial={theta_initial}',
        showlegend=True, width=1500, height=600
    )
    fig_v.update_xaxes(title_text="k")
    fig_v.update_yaxes(tickformat=".3f", title_text="v_k")

    # File-safe suffix (avoid dots in numeric value)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Save PNGs into the figures folder
    fig_theta.write_image(os.path.join(figures_dir, f"adam_theta_ncvx_{suffix}.png"))
    fig_m.write_image(os.path.join(figures_dir, f"adam_m_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"adam_v_ncvx_{suffix}.png"))

    print(f"Saved figures (theta_initial={theta_initial}) in the folder '{figures_dir}'")



Adam Configuration: {'beta1': 0.0, 'beta2': 0.99, 'lr': 0.1}, theta_initial=0.1
Epoch: 50, Error: 0.0.
Epoch: 100, Error: 0.0.
Last epoch: 100, Error: 0.0.

Adam Configuration: {'beta1': 0.0, 'beta2': 0.999, 'lr': 0.1}, theta_initial=0.1
Epoch: 50, Error: 0.0.
Epoch: 100, Error: 0.0.
Last epoch: 100, Error: 0.0.

Adam Configuration: {'beta1': 0.9, 'beta2': 0.99, 'lr': 0.1}, theta_initial=0.1
Epoch: 50, Error: 0.010280551015027117.
Epoch: 100, Error: 0.0007297613896508626.
Last epoch: 100, Error: 0.0007297613896508626.

Adam Configuration: {'beta1': 0.9, 'beta2': 0.999, 'lr': 0.1}, theta_initial=0.1
Epoch: 50, Error: 0.009430899233800583.
Epoch: 100, Error: 0.00043504334727240135.
Last epoch: 100, Error: 0.00043504334727240135.

Adam Configuration: {'beta1': 0.0, 'beta2': 0.99, 'lr': 0.01}, theta_initial=0.1
Epoch: 50, Error: 0.010627653729664233.
Epoch: 100, Error: 0.0009613644870045679.
Epoch: 150, Error: 6.664613067419722e-06.
Epoch: 200, Error: 4.594908342703263e-09.
Last epoch: 20

## Nonlocal Adam

In [5]:
# --- Nonlocal Continuous Adam | Save PNG per initial condition ---
inits = [0.1,0,-0.1]

for theta_initial in inits:
    # 1) Create figures for this initial condition
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_m = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Figure titles specific to this initial condition
    fig_theta.update_layout(
        title_text=f'Theta values convergence — Nonlocal Continuous Adam (theta_initial={theta_initial})'
    )
    fig_m.update_layout(
        title_text=f'First moment (m) — Nonlocal Continuous Adam (theta_initial={theta_initial})'
    )
    fig_v.update_layout(
        title_text=f'Second moment (v) — Nonlocal Continuous Adam (theta_initial={theta_initial})'
    )

    # 2) Loop over learning rates and parameter combinations
    for i, lr in enumerate(param_grid['lr']):
        # Number of epochs per lr
        if lr == 0.1:
            epochs = 100
        elif lr == 0.01:
            epochs = 200

        # Continuous time span for the solver
        t = [0.0, epochs * lr]

        # All parameter sets with this lr
        filtered_params = [p for p in param_list if p['lr'] == lr]
        for params in filtered_params:
            print(f'\nNonlocal Continuous Adam Configuration: {params}, theta_initial={theta_initial}')

            # Initialize solver with this initial condition
            solver = NonlocalSolverMomentumAdam(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]),
                alpha=params['lr'], betas=[params['beta1'], params['beta2']]
            )
            t_values, y_values = solver.solve()

            label = f"beta1={params['beta1']}, beta2={params['beta2']}"

            # Theta(t) trajectory — x-axis normalized as t/alpha for comparison with k
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],          # normalized x-axis
                y=np.asarray(y_values).squeeze(),   # theta(t)
                mode='lines',
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            # m(t) and v(t) stored by the solver as two-column arrays [t, value]
            numerators   = np.asarray(solver._last_m)  # columns: [t, m]
            denominators = np.asarray(solver._last_v)  # columns: [t, v]

            fig_m.add_trace(go.Scatter(
                x=numerators[:, 0] / params['lr'],
                y=numerators[:, 1],
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            fig_v.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

    # 3) Axes and sizing
    fig_theta.update_xaxes(title_text="t/alpha")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta(t)")
    fig_theta.update_layout(width=1500, height=600)

    fig_m.update_xaxes(title_text="t/alpha")
    fig_m.update_yaxes(tickformat=".2f", title_text="m(t)")
    fig_m.update_layout(width=1500, height=600)

    fig_v.update_xaxes(title_text="t/alpha")
    fig_v.update_yaxes(tickformat=".3f", title_text="v(t)")
    fig_v.update_layout(width=1500, height=600)

    # 4) File-name suffix per initial condition (avoid dots)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # 5) Save all figures for THIS initial condition
    fig_theta.write_image(os.path.join(figures_dir, f"nonlocal_adam_theta_ncvx_{suffix}.png"))
    fig_m.write_image(os.path.join(figures_dir, f"nonlocal_adam_m_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"nonlocal_adam_v_ncvx_{suffix}.png"))

    print(f"Saved figures (theta_initial={theta_initial}) in the folder '{figures_dir}'")


Nonlocal Continuous Adam Configuration: {'beta1': 0.0, 'beta2': 0.99, 'lr': 0.1}, theta_initial=0.1
Iter 20 – err 17.712378562617328
Iter 40 – err 52.72446226964993
Iter 60 – err 6.021498693742585
Iter 80 – err 4.063861730938834
Iter 100 – err 8.351895132363325
Iter 120 – err 10.06761152449896
Iter 140 – err 6.188553461920138
Iter 160 – err 4.3596126682131215
Iter 180 – err 9.34024297186246
Iter 200 – err 6.971007816914885
Iter 220 – err 7.408428626365485
Iter 240 – err 7.142581801174794
Iter 260 – err 8.051112833525675
Iter 280 – err 8.820044081742875
Iter 300 – err 4.8970501916007425
Iter 320 – err 9.414325623250893
Iter 340 – err 5.167921925675664
Iter 360 – err 8.479463420574827
Iter 380 – err 6.52582031694053
Iter 400 – err 5.46387642653966
Iter 420 – err 2.4849415272695166
Iter 440 – err 5.936781774595907
Iter 460 – err 5.439166144623702
Iter 480 – err 0.7358370567769222
Iter 500 – err 0.132242379150488
Iter 520 – err 0.46877216252749626
Iter 540 – err 0.3111280770182047
Iter 56

## Both Models Together

In [6]:
# --- Adam (discrete) vs. Nonlocal Continuous Adam
# --- Separate PNGs per initial condition theta_initial ---

config_colors = {
    (0.9, 0.99): 'blue',
    (0.9, 0.999): 'green',
    (0.0, 0.99): 'red',
    (0.0, 0.999): 'purple'
}

inits = [0.1,0,-0.1]

for theta_initial in inits:
    # Create figures for this initial condition
    fig_theta = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_m = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Iterate over each learning rate
    for i, lr in enumerate(param_grid['lr']):

        # Filter parameter sets for this lr
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs and t-span 
        if lr == 0.1:
            epochs = 100
            t_inicial = lr
        elif lr == 0.01:
            epochs = 200
            t_inicial = lr

        # Continuous-time span for the nonlocal solver
        t = [t_inicial, epochs * lr]

        # -------- Discrete Adam --------
        for params in filtered_params:
            print(f'\nAdam Configuration: {params}, theta_initial={theta_initial}')

            solver = AdamMomentum(
                dL=dL, lr=lr, beta1=params['beta1'], beta2=params['beta2'], epochs=epochs
            )
            solver.solve(theta_initial=theta_initial)

            color = config_colors[(params['beta1'], params['beta2'])]

            # θ_k (discrete trajectory)
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.01), color=color),
                name=f'Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # m_k (first moment)
            fig_m.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.m_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.01), color=color),
                name=f'Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v_k (second moment)
            fig_v.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.v_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.01), color=color),
                name=f'Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

        # ---- Nonlocal continuous-time Adam ----
        for params in filtered_params:
            color = config_colors[(params['beta1'], params['beta2'])]
            print(f'\nNonlocal Continuous Adam Configuration: {params}, theta_initial={theta_initial}')

            solver_nonlocal = NonlocalSolverMomentumAdam(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]),
                alpha=params['lr'], betas=[params['beta1'], params['beta2']]
            )
            t_values, y_values = solver_nonlocal.solve()

            # theta(t) (continuous trajectory)
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],
                y=y_values,
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Nonlocal Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # m(t) and v(t) from the continuous solver
            numerators   = np.asarray(solver_nonlocal._last_m)  # [t, m]
            denominators = np.asarray(solver_nonlocal._last_v)  # [t, v]

            fig_m.add_trace(go.Scatter(
                x=numerators[:, 0] / params['lr'],
                y=numerators[:, 1],
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Nonlocal Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            fig_v.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Nonlocal Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

    # Labels and formatting
    for fig, ytxt in [
        (fig_theta, "Theta values"),
        (fig_m, "First moment values"),
        (fig_v, "Second moment values")
    ]:
        fig.update_xaxes(title_text="t/alpha")
        if ytxt == "Theta values":
            fig.update_yaxes(tickformat=".1f", title_text=ytxt)
        elif ytxt == "First moment values":
            fig.update_yaxes(tickformat=".2f", title_text=ytxt)
        else:
            fig.update_yaxes(tickformat=".3f", title_text=ytxt)
        fig.update_layout(width=1500, height=600, showlegend=True)

    fig_theta.update_layout(title_text=f"Theta values convergence trajectories — theta_initial={theta_initial}")
    fig_m.update_layout(title_text=f"First moment (m) trajectories — theta_initial={theta_initial}")
    fig_v.update_layout(title_text=f"Second moment (v) trajectories — theta_initial={theta_initial}")

    # File suffix (avoid dots)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Save PNGs for this initial condition
    fig_theta.write_image(os.path.join(figures_dir, f"adam_vs_nonlocal_theta_ncvx_{suffix}.png"))
    fig_m.write_image(os.path.join(figures_dir, f"adam_vs_nonlocal_m_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"adam_vs_nonlocal_v_ncvx_{suffix}.png"))

    print(f"Saved figures (theta_initial={theta_initial}) in the folder '{figures_dir}'")



Adam Configuration: {'beta1': 0.0, 'beta2': 0.99, 'lr': 0.1}, theta_initial=0.1
Epoch: 50, Error: 0.0.
Epoch: 100, Error: 0.0.
Last epoch: 100, Error: 0.0.

Adam Configuration: {'beta1': 0.0, 'beta2': 0.999, 'lr': 0.1}, theta_initial=0.1
Epoch: 50, Error: 0.0.
Epoch: 100, Error: 0.0.
Last epoch: 100, Error: 0.0.

Adam Configuration: {'beta1': 0.9, 'beta2': 0.99, 'lr': 0.1}, theta_initial=0.1
Epoch: 50, Error: 0.010280551015027117.
Epoch: 100, Error: 0.0007297613896508626.
Last epoch: 100, Error: 0.0007297613896508626.

Adam Configuration: {'beta1': 0.9, 'beta2': 0.999, 'lr': 0.1}, theta_initial=0.1
Epoch: 50, Error: 0.009430899233800583.
Epoch: 100, Error: 0.00043504334727240135.
Last epoch: 100, Error: 0.00043504334727240135.

Nonlocal Continuous Adam Configuration: {'beta1': 0.0, 'beta2': 0.99, 'lr': 0.1}, theta_initial=0.1
Iter 20 – err 17.305807835690302
Iter 40 – err 52.844625480190814
Iter 60 – err 4.568963750085856
Iter 80 – err 12.022892945311312
Iter 100 – err 9.2268134757643